# Revisão do M9

Este notebook audita a camada associativa da RQ3 após as reformulações de M6--M8.

**Conclusão:** M9 não deve tratar ausência de commits como nota de planejamento nem depender de um único score LLM. A análise recomendada associa componentes observáveis separados, estratifica a razão de retrabalho pela elegibilidade de baseline e expõe influência de casos, sem alegações causais.

## Estrutura recomendada

| Associação | Unidade e estrato | Finalidade |
|---|---|---|
| **M9a - Escopo estrutural T1 vs. retrabalho limpo** | 14 equipe-semestres para magnitude; 9 elegíveis para razão | Testar evidência estrutural sem imputar ausência |
| **M9b - Escopo estrutural T1 vs. avaliação T3** | 14 equipe-semestres | Contrastar artefatos iniciais e resultados independentes |
| **M9c - Diagnóstico de influência** | Mesmo conjunto da associação | Mostrar sensibilidade leave-one-out, não significância |
| **M9d - Conteúdo declarativo vs. desfechos** | Somente após M6b estruturado e auditável | Extensão futura; não usar o score legado opaco |

M9 não cria um indicador novo de planejamento ou retrabalho. Ele conecta M6a/M6b, M8a/M8b e avaliações T3 sem colapsar construtos distintos.

## 1. Vínculo com a RQ3

O notebook confirma no texto atual do paper que M9 pertence à RQ3 e falha se a posição da métrica mudar.

In [ ]:
from hashlib import sha256
from pathlib import Path
import re

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'paper_v8/latex_code/main.tex').is_file():
            return candidate
    raise FileNotFoundError('Could not locate project root')

PROJECT_ROOT = find_project_root(Path.cwd())
PAPER_PATH = PROJECT_ROOT / 'paper_v8/latex_code/main.tex'
LEGACY_CORRELATION_PATH = PROJECT_ROOT / 'paper_v8/data/m9_planning_vs_rework_association.csv'
LEGACY_GROUP_PATH = PROJECT_ROOT / 'paper_v8/data/m9_planning_vs_rework_association_group_contrast.csv'
PLANNING_PATH = PROJECT_ROOT / 'data/analysis/planning_metrics.parquet'
REWORK_PATH = PROJECT_ROOT / 'paper_v8/data/m8_rework_severity_ratio.csv'
OUTCOMES_PATH = PROJECT_ROOT / 'data/analysis/cross_evidence/datasets/evaluator_outcome_metrics.parquet'

paper_text = PAPER_PATH.read_text(encoding='utf-8')
rq_matches = dict(re.findall(r'\\item \\textbf\{(RQ\d)[^}]*:\}\s*(.*?)\n', paper_text))
m9_position = paper_text.index(r'\textbf{M9 --')
rq3_position = paper_text.index(r'\subsubsection{RQ3:')
detected_rq = 'RQ3' if rq3_position < m9_position else 'UNKNOWN'
assert detected_rq == 'RQ3'
assert rq_matches[detected_rq]
pd.DataFrame([{'metric':'M9','detected_rq':detected_rq,'rq_text':rq_matches[detected_rq],'paper_sha256':sha256(paper_text.encode()).hexdigest()}])

In [ ]:
CONFIG = {
    'metric': 'M9',
    'expected_rq': 'RQ3',
    'unit_of_analysis': 'team_semester',
    'planning_predictors': ['pi_file_count_t1', 'planning_scope_log1p_t1'],
    'rework_outcomes': ['clean_rework_churn_t3', 'clean_rework_ratio_t3'],
    'ratio_eligibility': 'prior_clean_path_t1_or_t2',
    'evaluator_outcomes': ['project_progress_mean_t3','scope_applicability_mean_t3','technical_complexity_mean_t3','engagement_participation_mean_t3'],
    'missingness_policy': 'stratify_and_report_no_score_floor_imputation',
    'influence_analysis': 'leave_one_out_spearman',
    'inference': 'exploratory_descriptive',
    'm6b_status': 'unavailable_pending_structured_reprocessing',
}
assert CONFIG['expected_rq'] == detected_rq
assert CONFIG['missingness_policy'] == 'stratify_and_report_no_score_floor_imputation'
CONFIG

## 2. Auditoria do M9 legado

O M9 publicado associa a nota LLM M6 aos valores M8 e a desfechos dos avaliadores. A versão de sensibilidade atribui nota 1 a equipes sem atividade T1. Essa imputação não é válida após M7: ausência de Git não identifica ausência de planejamento. A auditoria preserva os resultados legados como linha de base, mas não os reutiliza como desenho principal.

In [ ]:
legacy_correlations = pd.read_csv(LEGACY_CORRELATION_PATH)
legacy_groups = pd.read_csv(LEGACY_GROUP_PATH)
assert set(legacy_correlations.columns) == {'outcome','n','spearman_rho','spearman_p'}
assert legacy_correlations['n'].eq(9).all()
assert set(legacy_groups.planning_group) == {'high','low','omitted'}
legacy_audit = pd.DataFrame([
 ('legacy_predictor','t1_planning_score from commit-subject LLM'),
 ('legacy_complete_case_n',int(legacy_correlations.n.iloc[0])),
 ('legacy_omitted_group_n',int(legacy_groups.loc[legacy_groups.planning_group.eq('omitted'),'n_total'].iloc[0])),
 ('floor_imputation_valid','no: M7 identifies repository inactivity, not planning absence'),
],columns=['check','value'])
legacy_audit

## 3. Associação estrutural regenerada

M9a/M9b usam M6a: contagem de artefatos T1 e escopo logarítmico de linhas em artefatos de planejamento. M8a é a magnitude limpa de retrabalho; M8b é a razão limpa. A razão só é interpretada para equipes com ao menos um caminho limpo de T1/T2, pois as demais não tinham baseline observável para revisão.

In [ ]:
keys=['ID_Equipe','Semestre']
planning=pd.read_parquet(PLANNING_PATH)
rework=pd.read_csv(REWORK_PATH,dtype={'Semestre':str})
outcomes=pd.read_parquet(OUTCOMES_PATH)
required_planning={'pi_file_count_t1','pi_line_delta_t1'}
assert not (required_planning-set(planning))
analysis_frame=(planning[keys+['pi_file_count_t1','pi_line_delta_t1']].merge(rework,on=keys,validate='one_to_one').merge(outcomes,on=keys,validate='one_to_one'))
analysis_frame['planning_scope_log1p_t1']=np.log1p(analysis_frame.pi_line_delta_t1)
analysis_frame['prior_clean_path_n']=analysis_frame.pi_file_count_t1
analysis_frame['baseline_eligible_for_rework']=analysis_frame.prior_clean_path_n.gt(0)
analysis_frame=analysis_frame.rename(columns={'rework_churn_t3':'clean_rework_churn_t3','rework_ratio_t3':'clean_rework_ratio_t3'})
assert len(analysis_frame)==14
assert analysis_frame.baseline_eligible_for_rework.sum()==9
analysis_frame[keys+CONFIG['planning_predictors']+CONFIG['rework_outcomes']+['baseline_eligible_for_rework']].sort_values(['Semestre','ID_Equipe'])

In [ ]:
def association_table(frame: pd.DataFrame, predictors: list[str], outcomes: list[str], analysis_id: str) -> pd.DataFrame:
    rows=[]
    for predictor in predictors:
        for outcome in outcomes:
            pair=frame[[predictor,outcome]].dropna()
            if len(pair)<3 or pair[predictor].nunique()<2 or pair[outcome].nunique()<2:
                rho=p_value=np.nan
            else:
                rho,p_value=spearmanr(pair[predictor],pair[outcome])
            rows.append({'analysis_id':analysis_id,'predictor':predictor,'outcome':outcome,'n':len(pair),'spearman_rho':rho,'spearman_p_exploratory':p_value})
    return pd.DataFrame(rows)

rework_magnitude_associations=association_table(analysis_frame,CONFIG['planning_predictors'],['clean_rework_churn_t3'],'all_team_semesters')
rework_ratio_associations=association_table(analysis_frame.loc[analysis_frame.baseline_eligible_for_rework],CONFIG['planning_predictors'],['clean_rework_ratio_t3'],'baseline_eligible_only')
evaluator_associations=association_table(analysis_frame,CONFIG['planning_predictors'],CONFIG['evaluator_outcomes'],'all_team_semesters')
revised_associations=pd.concat([rework_magnitude_associations,rework_ratio_associations,evaluator_associations],ignore_index=True)
assert set(revised_associations.analysis_id)=={'all_team_semesters','baseline_eligible_only'}
revised_associations.round(3)

## 4. Diagnóstico de influência

Com $n=14$ ou $n=9$, um caso pode alterar substancialmente a associação. A análise leave-one-out mostra o intervalo de $
ho$ obtido ao remover cada equipe uma vez. Isso é diagnóstico de fragilidade, não teste de significância adicional.

In [ ]:
def leave_one_out_spearman(frame: pd.DataFrame, predictor: str, outcome: str, analysis_id: str) -> pd.DataFrame:
    pair=frame[keys+[predictor,outcome]].dropna().copy()
    rows=[]
    for row in pair.itertuples(index=False):
        remaining=pair.loc[~((pair.ID_Equipe==row.ID_Equipe)&(pair.Semestre==row.Semestre))]
        rho,p_value=spearmanr(remaining[predictor],remaining[outcome]) if len(remaining)>=3 else (np.nan,np.nan)
        rows.append({'analysis_id':analysis_id,'predictor':predictor,'outcome':outcome,'removed_team':row.ID_Equipe,'removed_semester':row.Semestre,'n_remaining':len(remaining),'spearman_rho':rho,'spearman_p_exploratory':p_value})
    return pd.DataFrame(rows)

loo_frames=[]
for predictor in CONFIG['planning_predictors']:
    loo_frames.append(leave_one_out_spearman(analysis_frame,predictor,'clean_rework_churn_t3','all_team_semesters'))
    loo_frames.append(leave_one_out_spearman(analysis_frame.loc[analysis_frame.baseline_eligible_for_rework],predictor,'clean_rework_ratio_t3','baseline_eligible_only'))
leave_one_out=pd.concat(loo_frames,ignore_index=True)
loo_summary=(leave_one_out.groupby(['analysis_id','predictor','outcome'],as_index=False).agg(n_removed=('removed_team','size'),rho_min=('spearman_rho','min'),rho_max=('spearman_rho','max')))
assert leave_one_out.groupby(['analysis_id','predictor','outcome']).size().min()>=9
loo_summary.round(3)

## 5. Não redundância e decisão

M9 é necessário como camada de associação, mas não deve reproduzir M6/M8 como um score composto. M9a/M9b ligam escopo de artefatos a magnitude/participação de retrabalho limpo; M9c expõe influência; M9d permanece indisponível até que M6b seja regenerado com extração estruturada e citações literais. M7 não entra como preditor, pois mede inatividade Git e é derivado da mesma ausência estrutural de M6a.

In [ ]:
m9_decision=pd.DataFrame([
 ('M9 legacy','retire as primary analysis','single opaque LLM predictor and invalid floor imputation for repository inactivity'),
 ('M9a','adopt','structural T1 planning scope versus clean rework magnitude across all 14 teams'),
 ('M9b','adopt with stratum','structural T1 planning scope versus clean rework ratio only among 9 baseline-eligible teams'),
 ('M9c','adopt','leave-one-out interval accompanies every descriptive association'),
 ('M9d','defer','textual planning-content association only after auditable M6b reprocessing'),
 ('M7','exclude as predictor','repository inactivity is a coverage/dynamic diagnostic, not independent planning evidence'),
],columns=['component','decision','reason'])
assert CONFIG['m6b_status']=='unavailable_pending_structured_reprocessing'
assert CONFIG['inference']=='exploratory_descriptive'
m9_decision

In [ ]:
assert detected_rq=='RQ3'
assert len(analysis_frame)==14
assert analysis_frame.baseline_eligible_for_rework.sum()==9
assert not revised_associations.empty
assert leave_one_out['n_remaining'].min()>=8
assert CONFIG['missingness_policy']=='stratify_and_report_no_score_floor_imputation'
m9_evidence_manifest={'metric':'M9','rq':detected_rq,'unit_of_analysis':CONFIG['unit_of_analysis'],'planning_source':str(PLANNING_PATH.relative_to(PROJECT_ROOT)),'rework_source':str(REWORK_PATH.relative_to(PROJECT_ROOT)),'outcome_source':str(OUTCOMES_PATH.relative_to(PROJECT_ROOT)),'team_semester_n':int(len(analysis_frame)),'baseline_eligible_n':int(analysis_frame.baseline_eligible_for_rework.sum()),'missingness_policy':CONFIG['missingness_policy'],'influence_analysis':CONFIG['influence_analysis'],'m6b_status':CONFIG['m6b_status'],'inference':CONFIG['inference']}
m9_evidence_manifest